# Part 1: Build and save the RAG vector store

**This notebook** reads **`RAG_docs/knowledge/`** (PDFs and optional JSON) and writes **`RAG_docs/vector_store/`** (FAISS + `rag_manifest.json`) plus **`rag_parents.json`**. Run **`Plan.ipynb`** (Part 2) after a successful build.

## Index build (what runs before save)

- **Section merge**: PDF pages merged into heading-based sections.
- **Semantic parents**: paragraph-embedding similarity splits each section into parent chunks.
- **Retrieval title**: one title per parent (default: local HF summarizer; or extractive / LLM via `TITLE_MODE` in the code cell).
- **Child chunks + FAISS**: small chunks for search; each indexed vector is **`retrieval_title` + child body** (batched embedding via helpers where applicable).
- **Sidecar**: `rag_parents.json` stores full parent text and metadata so Part 2 can expand child hits to full sections.

### PDFs, semantic parents, and retrieval titles

| Step | What happens |
|------|----------------|
| **Section merge** | Per-page PDF rows are merged into one item per file with **`_pdf_sections`** (outline/heading-based slices in `utils/rag_index_build.py`). |
| **Semantic parents** | Each section body is split into **parent** chunks using **paragraph embedding similarity** (boundaries where adjacent paragraphs are dissimilar). |
| **Retrieval title** | **One semantic / search-oriented title per parent** (not per PDF page): default local HF summarization; or **`TITLE_MODE_EXTRACTIVE`** (keyword line) / **`TITLE_MODE_LLM`**. Titles are printed while indexing. |
| **Child chunks + FAISS** | Parents are split into smaller **child** chunks; each indexed vector’s `page_content` is **`retrieval_title` + child text** so queries align with the title. |
| **Sidecar** | **`rag_parents.json`** stores **`parent_id`** → full parent body, **`retrieval_title`**, section heading, and pages so Part 2 can expand context beyond the child snippet. |

**JSON knowledge** (`.json` in `knowledge/`): each record is one logical section; it follows the **same** semantic-parent → retrieval-title → child-chunk path as PDF sections (see `json_kb_item_to_sections` in `utils/rag_index_build.py`).

**Titles (default):** local **Hugging Face summarization** (`sshleifer/distilbart-cnn-12-6`, Transformer seq2seq — no OpenAI). Override with `TITLE_MODE`: `"extractive"` (keywords) or `"llm"` (needs `OPENAI_API_KEY`). Env `RAG_SUMMARY_MODEL` sets another HF model id.

Requires **`pypdf`** for PDFs. Re-run whenever knowledge files change, then run **`Plan.ipynb`** (Part 2).

**Shared helpers:** `utils/rag_index_build.py`, `utils/rag_utils.py`. Re-run when KB files or chunking/embed settings change.

## Where the vector store is created

The **in-memory** FAISS index is built with **`faiss_from_documents_batched`** (chunked embedding) — one vector per **child** chunk. Indexed **`page_content`** is always **`retrieval_title` + child body** (the title is the **per–semantic-parent** line from indexing; see table above). The same applies to **JSON** and **PDF** sources after parent/title generation.

**`save_vector_store`** writes the FAISS folder + `rag_manifest.json`. **`save_parent_store`** writes **`rag_parents.json`** (parent bodies, **retrieval titles**, section metadata). Part 2 loads parents via **`utils.rag_utils.load_parent_store`** to optionally inject full parent text into the LLM context.

In [ ]:
import sys, warnings
if sys.version_info >= (3, 14):
    warnings.filterwarnings(
        "ignore",
        message="Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.",
        category=UserWarning,
    )

import json
from pathlib import Path
from typing import Any, Dict, List, Optional

from langchain_community.document_loaders import PyPDFLoader
from langchain_community.embeddings import SentenceTransformerEmbeddings
from utils.rag_index_build import (
    CHILD_CHUNK_OVERLAP,
    CHILD_CHUNK_SIZE,
    TITLE_MODE_EXTRACTIVE,
    TITLE_MODE_LLM,
    TITLE_MODE_TRANSFORMER,
    build_documents_from_knowledge_base,
    expand_pdf_kb_items_with_sections,
    save_parent_store,
)
from utils.compute_device import sentence_transformer_model_kwargs
from utils.rag_utils import faiss_from_documents_batched, save_vector_store
from utils.project_env import load_project_environment

load_project_environment()

try:
    import pypdf  # noqa: F401 — PyPDFLoader depends on this at runtime
except ImportError as e:
    raise ImportError(
        "PDF loading requires pypdf. Install with: pip install pypdf"
    ) from e

KNOWLEDGE_DIR = Path("RAG_docs/knowledge")
VECTOR_STORE_DIR = Path("RAG_docs/vector_store")
EMBED_MODEL = "all-MiniLM-L6-v2"
# sentence-transformers encode batch — set 8 or 4 if PyTorch reports CPU OOM
EMBED_ENCODE_BATCH = 16
# "transformer" = local HF summarizer (default) | "extractive" | "llm"
TITLE_MODE = TITLE_MODE_TRANSFORMER
SUMMARIZATION_MODEL = None  # or e.g. "sshleifer/distilbart-cnn-12-6"; env RAG_SUMMARY_MODEL overrides
PRINT_TITLES = True
MAX_LLM_PARENTS = 120  # only when TITLE_MODE == TITLE_MODE_LLM


def _normalize_kb_record(
    item: Dict[str, Any],
    source_file: str,
    doc_type: str,
    page: Optional[int] = None,
) -> Dict[str, Any]:
    out = dict(item)
    out["source_file"] = source_file
    out["doc_type"] = doc_type
    if page is not None:
        out["page"] = page
    return out


def load_pdf_file(file_path: Path) -> List[Dict[str, Any]]:
    """One dict per PDF page with title, text, source_file, doc_type, page."""
    loader = PyPDFLoader(str(file_path))
    docs = loader.load()
    rows: List[Dict[str, Any]] = []
    for i, doc in enumerate(docs):
        meta_page = doc.metadata.get("page")
        page_1based = int(meta_page) + 1 if meta_page is not None else i + 1
        rows.append(
            _normalize_kb_record(
                {
                    "title": f"{file_path.name} (page {page_1based})",
                    "text": doc.page_content or "",
                },
                file_path.name,
                "pdf",
                page=page_1based,
            )
        )
    return rows


def load_knowledge_base(knowledge_dir: Path, verbose: bool = True) -> List[Dict[str, Any]]:
    """Load all *.json and *.pdf into flat records with title, text, source_file, doc_type, page (PDF)."""
    kb: List[Dict[str, Any]] = []
    knowledge_json = sorted(knowledge_dir.glob("*.json"))
    knowledge_pdf = sorted(knowledge_dir.glob("*.pdf"))
    if not knowledge_json and not knowledge_pdf:
        raise FileNotFoundError(f"No JSON or PDF files found in {knowledge_dir}")

    if verbose:
        print(f"Loading knowledge base from {knowledge_dir}:")

    for knowledge_file in knowledge_json:
        filename = knowledge_file.name
        try:
            with open(knowledge_file, "r", encoding="utf-8") as f:
                data = json.load(f)
            if isinstance(data, list):
                n = 0
                for row in data:
                    if isinstance(row, dict) and "title" in row and "text" in row:
                        kb.append(_normalize_kb_record(row, filename, "json"))
                        n += 1
                if verbose:
                    print(f"  Loaded {n} document(s) from {filename}")
            elif isinstance(data, dict) and "title" in data and "text" in data:
                kb.append(_normalize_kb_record(data, filename, "json"))
                if verbose:
                    print(f"  Loaded 1 document from {filename}")
            elif verbose:
                print(f"  Skipped {filename} (unknown format)")
        except Exception as e:
            if verbose:
                print(f"  Error loading {filename}: {e}")

    for pdf_file in knowledge_pdf:
        try:
            docs = load_pdf_file(pdf_file)
            kb.extend(docs)
            if verbose:
                print(f"  Loaded {len(docs)} document(s) from {pdf_file.name} (PDF)")
        except Exception as e:
            if verbose:
                print(f"  Error loading {pdf_file.name}: {e}")

    if verbose:
        print(f"Total: {len(kb)} knowledge documents")
    return kb


# --- PDF pages → sections → semantic parents → child chunks → FAISS -----------------
knowledge_base = load_knowledge_base(KNOWLEDGE_DIR)
knowledge_expanded = expand_pdf_kb_items_with_sections(
    knowledge_base, KNOWLEDGE_DIR, verbose=True
)
embeddings = SentenceTransformerEmbeddings(
    model_name=EMBED_MODEL,
    model_kwargs=sentence_transformer_model_kwargs(),
    encode_kwargs={"batch_size": EMBED_ENCODE_BATCH},
)
rag_documents, parent_store = build_documents_from_knowledge_base(
    knowledge_expanded,
    embeddings,
    title_mode=TITLE_MODE,
    summarization_model=SUMMARIZATION_MODEL,
    print_titles=PRINT_TITLES,
    max_llm_parents=MAX_LLM_PARENTS,
    verbose=True,
)

print(
    f"Embedding model={EMBED_MODEL}, child chunk={CHILD_CHUNK_SIZE}/{CHILD_CHUNK_OVERLAP}, "
    f"{len(rag_documents)} FAISS vectors, {len(parent_store)} parents"
)

vector_store = faiss_from_documents_batched(rag_documents, embeddings)

save_vector_store(
    vector_store,
    VECTOR_STORE_DIR,
    knowledge_expanded,
    embed_model=EMBED_MODEL,
    chunk_size=CHILD_CHUNK_SIZE,
    chunk_overlap=CHILD_CHUNK_OVERLAP,
    n_chunks=len(rag_documents),
    extra_manifest={
        "indexing": "parent_child_semantic",
        "n_parents": len(parent_store),
        "title_mode": TITLE_MODE,
        "max_llm_parents": MAX_LLM_PARENTS,
    },
)
ps_path = save_parent_store(VECTOR_STORE_DIR, parent_store)
print(f"Saved vector store to {VECTOR_STORE_DIR}")
print(f"Saved parent store to {ps_path}")

Loading knowledge base from RAG_docs\knowledge:


Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 51 0 (offset 0)
Ignoring wrong pointing object 98 0 (offset 0)
Ignoring wrong pointing object 101 0 (offset 0)
Ignoring wrong pointing object 103 0 (offset 0)
Ignoring wrong pointing object 117 0 (offset 0)
Ignoring wrong pointing object 119 0 (offset 0)
Ignoring wrong pointing object 121 0 (offset 0)
Ignoring wrong pointing object 130 0 (offset 0)
Ignoring wrong pointing object 142 0 (offset 0)
Ignoring wrong pointing object 173 0 (offset 0)
Ignoring wrong pointing object 225 0 (offset 0)
Ignoring wrong pointing object 243 0 (offset 0)
Ignoring wrong pointing object 275 0 (offset 0)
Ignoring wrong pointing object 312 0 (offset 0)


  Loaded 9 document(s) from 20220311114640pmwebology186-196pdf21.pdf (PDF)
  Loaded 46 document(s) from ATTACK_Design_and_Philosophy_March_2020.pdf (PDF)
  Loaded 146 document(s) from CIS_Controls_Guide_v8.1.2_0325_v2.pdf (PDF)
  Loaded 21 document(s) from cisos-guide-to-the-top-cybersecurity-frameworks.pdf (PDF)
  Loaded 44 document(s) from iso-27001.pdf (PDF)
  Error loading MITRE_ATT&CK_and_Building_Better_Defenses.pdf: cryptography>=3.1 is required for AES algorithm
  Loaded 59 document(s) from NIST.SP.800-207.pdf (PDF)
  Loaded 492 document(s) from NIST.SP.800-53r5.pdf (PDF)
  Loaded 165 document(s) from open_ran_security_report_full_report_0.pdf (PDF)


Ignoring wrong pointing object 9 0 (offset 0)
Ignoring wrong pointing object 13 0 (offset 0)
Ignoring wrong pointing object 15 0 (offset 0)
Ignoring wrong pointing object 19 0 (offset 0)
Ignoring wrong pointing object 21 0 (offset 0)
Ignoring wrong pointing object 23 0 (offset 0)
Ignoring wrong pointing object 51 0 (offset 0)
Ignoring wrong pointing object 98 0 (offset 0)
Ignoring wrong pointing object 101 0 (offset 0)
Ignoring wrong pointing object 103 0 (offset 0)
Ignoring wrong pointing object 117 0 (offset 0)
Ignoring wrong pointing object 119 0 (offset 0)
Ignoring wrong pointing object 121 0 (offset 0)
Ignoring wrong pointing object 130 0 (offset 0)
Ignoring wrong pointing object 142 0 (offset 0)
Ignoring wrong pointing object 173 0 (offset 0)
Ignoring wrong pointing object 225 0 (offset 0)
Ignoring wrong pointing object 243 0 (offset 0)
Ignoring wrong pointing object 275 0 (offset 0)
Ignoring wrong pointing object 312 0 (offset 0)


  Loaded 12 document(s) from paper3.pdf (PDF)
Total: 994 knowledge documents
  PDF sections: 20220311114640pmwebology186-196pdf21.pdf -> 11 section(s)
  PDF sections: ATTACK_Design_and_Philosophy_March_2020.pdf -> 36 section(s)
  PDF sections: CIS_Controls_Guide_v8.1.2_0325_v2.pdf -> 8 section(s)
  PDF sections: NIST.SP.800-207.pdf -> 126 section(s)
  PDF sections: NIST.SP.800-53r5.pdf -> 970 section(s)
  PDF sections: cisos-guide-to-the-top-cybersecurity-frameworks.pdf -> 17 section(s)
  PDF sections: iso-27001.pdf -> 58 section(s)
  PDF sections: open_ran_security_report_full_report_0.pdf -> 351 section(s)
  PDF sections: paper3.pdf -> 24 section(s)


...

  [title 2057] 'open_ran_security_report_full_report_0.pdf' | p_a2b91bbae71de6db | open_ran_security_report_full_report_0: ISO/IEC 5962:2021 -  Information technology —                   SPDX® Specification V2.2.1,  published August 2021 . The SBOM must be provided in Software Package Data (SPDX) or Software Identification (SWID) format .


Your max_length is set to 90, but your input_length is only 46. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=23)


  [title 2058] 'open_ran_security_report_full_report_0.pdf' | p_78967838f4c016f9 | open_ran_security_report_full_report_0: Report was commissioned and financed by the Ministry of Internal Affairs and Communications of Japan . Report was prepared in cooperation with industrial  partners including a MNO and cybersecurity companies .
  [title 2059] 'paper3.pdf' | p_52811b71af2da3c8 | paper3: Yevhenii Kurii1 and Ivan Opirskyy1 analyzed the NIST SP 800-53 and ISO/IEC 27001:2013 .
  [title 2060] 'paper3.pdf' | p_c0797aa443931f6a | paper3: The most commonly-found and preferred by security profe ssionals worldwide are NISTSP 800 -53 and ISO /IEC 27001:2013 . They combine both the comprehensive set of security controls to cover the most important security areas and wide applicability . But they also have a set of distinct featur
  [title 2061] 'paper3.pdf' | p_6bd7722a16769531 | paper3: A key consideration for choosing an information security framework would be understanding the level of conten

Your max_length is set to 90, but your input_length is only 82. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=41)


  [title 2074] 'paper3.pdf' | p_440a1bd7d73675ad | paper3: Privacy Program Plan Plan None Confidentiality,  . 17.                : .                                          ‘ż�� ’ ‚��-  Ś�‚-‚Ś-        “Credibility” is a “privacy program”, a ‚privacy” program, a program that
  [title 2075] 'paper3.pdf' | p_887c7635850eaa9e | paper3: Prime Minister PM PM: 22.                                                                      ‘   ’         ‚��   ;              Charity   nominee:  Ś�Śż�   vanishing disparity, precaution and anonymity .
  [title 2076] 'paper3.pdf' | p_315fa810b792c50b | paper3: A.G.A. 25.1.1, 20.2, 2025.1 . 25.2.3, 25.4.1 and 25.3.1: 20.5.3 .
  [title 2077] 'paper3.pdf' | p_f4ee480351c1dfe5 | paper3: 32  Purposing None Confidentiality, .                                                             ‚��                     - Categorized with uniquely preferred attributes .
  [title 2078] 'paper3.pdf' | p_653f154e40db3945 | paper3: A.15.2.2*   . 28.4.1.1, A.5.1 .1.2, .6.3, 8

In [ ]:
# Optional: smoke-test retrieval (edit query)
# L2 distance → heuristic score; metadata "retrieval_title" reflects TITLE_MODE (transformer / extractive / llm).
q = "DDoS mitigation NIST zero trust access control"
pairs = vector_store.similarity_search_with_score(q, k=5)
print(f"Query: {q!r} -> {len(pairs)} hits")
for i, (doc, dist) in enumerate(pairs, 1):
    rt = (doc.metadata.get("retrieval_title") or doc.metadata.get("title") or "")[:88]
    src = doc.metadata.get("source_file") or "?"
    pid = doc.metadata.get("parent_id") or "-"
    score = 1.0 / (1.0 + float(dist))
    print(f"  [{i}] {rt}... | src={src} parent={pid[:12]}... | score={score:.4f}")